In [ ]:
import pandas as pd
import numpy as np

from utils import bootstrap_quantile_conf_int

In [ ]:
measure = "CMCT (msec)"

In [ ]:
df = pd.read_csv(f"data/{measure}.csv")
df.head()

In [ ]:
# Reference limits 2.5 - 97.5 quantile by side/ muscle group

# Point estimate
ref_df = (
    df.groupby(by=["muscle", "side"])[["value"]]
    .quantile([0.025, 0.5, 0.975])
    .reset_index()
    .rename(columns={"level_2": "quantile"})
)

# Confidence intervals (via bootstrapping)
ci_summary = []
for (muscle, side), g in df.groupby(by=["muscle", "side"]):
    for q in [0.025, 0.5, 0.975]:
        ci_lo, ci_hi, n = bootstrap_quantile_conf_int(g["value"], q=q, n_boot=10_000)
        ci_summary.append(
            {
                "muscle": muscle,
                "side": side,
                "quantile": q,
                "ci_lower": ci_lo,
                "ci_upper": ci_hi,
                "n": n,
            }
        )

In [ ]:
# Join point est. + conf. ints

reference_summary_df = pd.merge(
    left=ref_df, right=pd.DataFrame(ci_summary), on=["muscle", "quantile", "side"]
)
reference_summary_df = reference_summary_df.pivot(
    index=["muscle", "quantile"],
    values=["value", "ci_lower", "ci_upper", "n"],
    columns="side",
).swaplevel(axis=1)

In [ ]:
right_ref_df = reference_summary_df["Right"]
right_ref_df.columns = ["right_est", "right_lower_95", "right_upper_95", "n_right"]
right_ref_df.tail(50)

In [ ]:
left_ref_df = reference_summary_df["Left"]
left_ref_df.columns = ["left_est", "left_lower_95", "left_upper_95", "n_left"]
left_ref_df.head()

In [ ]:
# Formatting for CIs

left_ref_df["left (CI)"] = (
    left_ref_df["left_est"].round(2).astype(str)
    + " ("
    + left_ref_df["left_lower_95"].round(2).astype(str)
    + ", "
    + left_ref_df["left_upper_95"].round(2).astype(str)
    + ")"
)
right_ref_df["right (CI)"] = (
    right_ref_df["right_est"].round(2).astype(str)
    + " ("
    + right_ref_df["right_lower_95"].round(2).astype(str)
    + ", "
    + right_ref_df["right_upper_95"].round(2).astype(str)
    + ")"
)

In [ ]:
estimate_summmary_df = left_ref_df.join(right_ref_df)
estimate_summmary_df = pd.melt(estimate_summmary_df, ignore_index=False)
estimate_summmary_df = estimate_summmary_df.reset_index().pivot(
    index=["muscle", "variable"], columns="quantile", values="value"
)
estimate_summmary_df.to_csv(f"results/{measure}/reference_limits.csv")
estimate_summmary_df

In [ ]:
# Asymmetry limits (one-sided - upper quantile only)
asymmetry_df_dropped_na = pd.read_csv(f"data/{measure}_asymmetry.csv")

In [ ]:
# Asymmetry reference by muscle group (right - left absolute value)

asym_ref_df = (
    asymmetry_df_dropped_na.groupby(by=["muscle"])[["r_minus_l_abs"]]
    .quantile([0.5, 0.95])
    .reset_index()
    .rename(columns={"level_1": "quantile"})
)

# Confidence intervals via bootstrapping
ci_summary = []
for muscle, g in asymmetry_df_dropped_na.groupby(by=["muscle"]):
    for q in [0.5, 0.95]:
        ci_lo, ci_hi, n = bootstrap_quantile_conf_int(
            g["r_minus_l_abs"], q=q, n_boot=10_000
        )
        ci_summary.append(
            {
                "muscle": muscle[0],
                "quantile": q,
                "ci_lower": ci_lo,
                "ci_upper": ci_hi,
                "n": n,
            }
        )

In [ ]:
# Save asymmetry limits

reference_summary_df = pd.merge(
    left=asym_ref_df, right=pd.DataFrame(ci_summary), on=["muscle", "quantile"]
)
reference_summary_df = pd.melt(
    reference_summary_df.set_index(["muscle", "quantile"]), ignore_index=False
)
reference_summary_df.reset_index().pivot(
    index=["muscle", "variable"], columns="quantile", values="value"
).to_csv(f"results/{measure}/asymmetry_reference_limits.csv")